# Lab 1: A RAG Pipeline over the Handbook and Circulars

**Time:** about 90 minutes

On Day 7 a model answered policy questions from memory and got some badly wrong. On Day 8, pasting the whole handbook
into the prompt helped, but that can't work for thousands of pages. Today's answer is **retrieval-augmented generation
(RAG)**: for each question, *find* the few passages that answer it, and give only those to the model.

Retrieval-augmented generation is simple to describe:

```
question ─► find the most relevant passages ─► give them to the LLM with strict instructions ─► answer with citations
              (retrieval)                        (augmentation)                                   (generation)
```

It's what a good new colleague does with a question they don't know the answer to: look it up in the handbook, read the
relevant paragraph, answer from it, and say where they found it.

Getting it *right* is where the work is. Most failures of real RAG systems aren't the fault of the LLM (large language
model). They're retrieval failures: the right passage was never found, or an **out-of-date** passage was found instead.
So you'll build each stage separately and measure it before moving on:

1. **Load** the documents with their metadata (which document, which section, what date).
2. **Chunk** them into passages, and compare two ways of doing it.
3. **Embed** the chunks and build a vector index.
4. **Measure retrieval** on the test questions before any LLM is involved.
5. Handle **superseded** rules.
6. **Generate** answers with citations, and refuse when nothing relevant was found.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **RAG** | retrieval-augmented generation: find relevant passages, then have the LLM answer from them |
| **metadata** | facts about a document: its name, section, date, what it replaces |
| **chunk** | one passage of a document, the unit that is stored and retrieved |
| **embedding** | a list of numbers representing a text's meaning; similar meanings get similar lists |
| **embedding model** | a model that turns text into embeddings (here, E5) |
| **vector index** | a store of embeddings that can quickly find the ones closest to a question's embedding |
| **FAISS** | Facebook AI Similarity Search: a library for fast nearest-vector search |
| **dense search** | retrieval by embedding similarity (meaning) |
| **keyword search** | retrieval by shared words (TF-IDF, Day 6) |
| **hybrid search** | combining dense and keyword search results |
| **hit@k** | did the right document appear in the top k results? |
| **MRR** | mean reciprocal rank: 1 if the right document came first, 1/2 if second, and so on |
| **superseded** | replaced by a newer rule |
| **relevance threshold** | the minimum similarity needed before the LLM is asked at all |
| **citation** | a numbered reference showing which passage an answer came from |
| **guardrail** | a check outside the model that stops certain questions reaching it |

## Where this lab fits

**What you've done so far.** On Day 6 you turned text into vectors (TF-IDF and Word2Vec) and extracted facts with rules.
On Day 7 you saw small LLMs answer policy from memory and get it wrong, and learned to make them answer only from
documents you paste in, with a refusal sentence. On Day 8 you found that fine-tuning changes a model's style more
reliably than its facts, and that pasting the whole handbook helped, but can't scale.

**What you'll do now.** You'll build retrieval-augmented generation (RAG) one stage at a time, measuring each before
moving on: load the documents with dates, cut them into chunks two ways, embed them with a model you've checked can
handle Swahili, measure retrieval on its own, add keyword search, remove superseded rules before the LLM ever sees them,
generate answers with citations behind a relevance threshold, add a guardrail for personal-record questions, test the
whole assistant end to end, and save everything for Lab 2.

**Why it matters.** This is the design that lets an LLM answer the Fund's policy questions safely: from the current
documents, with sources a staff member can check, and with a refusal when the documents don't have the answer. When a
circular changes a rule, you update the document and rebuild the index; nothing is retrained.

**What happens next.** Lab 2 wraps this pipeline in an internal web service that other Fund systems can call. Lab 3 works
out what it would cost to run on the Fund's own servers versus a cloud platform.

### Theory: two places a model can get knowledge from

| | Knowledge in the weights (Days 7-8) | Knowledge in documents (RAG) |
|---|---|---|
| Where facts live | learned into the model's billions of numbers | in a document library you control |
| Updating a rule | retrain or fine-tune | edit the document, rebuild the index |
| Can it say where an answer came from? | no | yes: it cites the passage |
| Behaviour when it doesn't know | often invents something plausible | can refuse when nothing relevant is found |
| What can go wrong | wrong or outdated facts, stated confidently | the right passage isn't found, or an outdated one is |

RAG doesn't make the model smarter. It changes the job: from *remembering* the Fund's rules to *reading* the right
paragraph and summarising it. Small models are much better at the second job. That's why most of this lab is about
**retrieval**: making sure the right, current paragraph is what the model reads.

## 0. Set up today's environment (run this cell first)

You don't need to create a folder or an environment by hand today. The cell below does it for you:

1. It finds the environment you created on Day 1 (`DAY1/Labs/work/fund-ai/.venv`). If it isn't there, it creates it.
2. It installs today's libraries into it from ``requirements.txt``. The first time takes a few minutes; after that, seconds.
3. If this machine has an NVIDIA GPU (graphics processing unit), it switches PyTorch to its GPU build, so training runs much faster.
4. It connects the environment to Jupyter as **Python (fund-ai)**, and tells you if this notebook isn't using it yet.

It's safe to run again at any time. If it prints instructions to switch the kernel, follow them and run it once more.

You don't need to read the code in this cell. It's housekeeping, not part of today's lesson.

In [ ]:
# ---- Setup for Day 9. Safe to run more than once. -------------------------------------------------------------
COURSE_ENV = None        # None = use the Day 1 environment. Or set a folder, e.g. Path.home() / "fund-ai-venv"
REQUIREMENTS = "requirements.txt"
EXTRAS = ['ollama:gemma2:2b']

import os, shutil, subprocess, sys
from pathlib import Path

labs = Path.cwd()
root = next((p for p in [labs, *labs.parents] if (p / "DAY1").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the course folder (the one that contains DAY1, DAY2, ...).")
env = Path(COURSE_ENV) if COURSE_ENV else root / "DAY1" / "Labs" / "work" / "fund-ai" / ".venv"
py = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(*cmd, quiet=False):
    """Run a command; show its output only if it fails."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if "path" in (r.stderr + r.stdout).lower() and os.name == "nt":
            print("\nIf the error mentions a path that is too long: Windows 'long paths' are switched off on this machine.\n"
                  "Ask ICT to enable them, or set COURSE_ENV above to a short folder, e.g. Path.home() / 'fund-ai-venv'.")
        raise RuntimeError("A setup step failed. The details are printed above.")
    return r


# 1. The course environment
if py.exists():
    print(f"- course environment found: {env}")
else:
    print(f"- creating the course environment (first time only): {env}")
    run(getattr(sys, "_base_executable", sys.executable), "-m", "venv", env)
    run(py, "-m", "pip", "install", "-q", "--upgrade", "pip")

# 2. Today's libraries
req = labs / REQUIREMENTS
print(f"- installing today's libraries from {REQUIREMENTS} ...")
run(py, "-m", "pip", "install", "-q", "-r", req)

# 3. Use the GPU if this machine has an NVIDIA one
if "torch" in req.read_text() and shutil.which("nvidia-smi"):
    if run(py, "-c", "import torch; print(torch.cuda.is_available())").stdout.strip() != "True":
        version = run(py, "-c", "import torch; print(torch.__version__.split('+')[0])").stdout.strip()
        print(f"- NVIDIA GPU found: switching PyTorch {version} to its GPU build (about 2.5 GB, first time only) ...")
        run(py, "-m", "pip", "uninstall", "-y", "-q", "torch")
        run(py, "-m", "pip", "install", "-q", f"torch=={version}", "--index-url", "https://download.pytorch.org/whl/cu124",
            "--extra-index-url", "https://pypi.org/simple")
    print("- PyTorch will use the GPU")

# 4. Anything else today needs
for extra in EXTRAS:
    kind, name = extra.split(":", 1)
    if kind == "spacy" and run(py, "-c", f"import {name}", quiet=True).returncode != 0:
        print(f"- downloading the spaCy model {name} ...")
        run(py, "-m", "spacy", "download", name)
    if kind == "ollama":
        import json, urllib.request
        try:
            tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))["models"]
            have = {m["name"] for m in tags}
            wanted = [m for m in name.split(",") if m]
            missing = [m for m in wanted if m not in have]
            print(f"- Ollama is running with {len(have)} model(s)" + (f"; still to pull: {', '.join(missing)}" if missing else ""))
            for m in missing:
                print(f"    in a terminal: ollama pull {m}")
        except OSError:
            print("- Ollama isn't running. Start the Ollama app (or run `ollama serve` in a terminal) before the LLM parts.")

# 5. Connect the environment to Jupyter
run(py, "-m", "ipykernel", "install", "--user", "--name", "fund-ai", "--display-name", "Python (fund-ai)")

# 6. Is this notebook using it?
if Path(sys.prefix).resolve() == env.resolve():
    print("\nReady. This notebook is running in the course environment. Carry on with the next cell.")
else:
    print("\nAlmost there. This notebook is running a different Python:")
    print(f"    {sys.executable}")
    print("Switch it to the course environment, then run this cell once more:")
    print("    1. Click the kernel name in the top right of the notebook, then 'Select Another Kernel'.")
    print("    2. Choose 'Jupyter Kernel' -> 'Python (fund-ai)'.")
    print("       (Or press Ctrl+Shift+P, run 'Python: Select Interpreter', choose 'Enter interpreter path...'")
    print("        and paste the path below, the same way you did on Day 1.)")
    print(f"    {py}")

## Imports and settings

The new tools today:

- `faiss`: the vector index, for fast nearest-vector search;
- `sentence_transformers`: loads embedding models. `SentenceTransformer` turns text into vectors; `util.cos_sim`
  measures how similar two vectors are;
- `TfidfVectorizer`: keyword search, as on Day 6.

`EMBED_MODEL` is **multilingual E5 small**, an embedding model that handles English and Swahili (you'll check that in
Section 3). The embeddings are computed on the GPU if there is one.

In [ ]:
import json
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import faiss                                                  # fast nearest-vector search
from sentence_transformers import SentenceTransformer, util   # embedding models
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
EMBED_MODEL = "intfloat/multilingual-e5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

Choose the LLM that will write the answers: the first model from a preference list that you have in Ollama (as in
Day 7 Lab 2).

In [ ]:
# Use the first of these LLMs that you have in Ollama.
installed = requests.get(OLLAMA_URL + "/api/tags").json()["models"]
available = []
for m in installed:
    available.append(m["name"])

LLM = available[0]
for name in ["gemma2:2b", "llama3.2:3b", "qwen2.5:3b", "qwen2.5:1.5b", "llama3.2:1b"]:
    if name in available:
        LLM = name
        break

print("embedding on", device, "| LLM:", LLM)

> **Where we are.** The libraries are loaded, the embedding model is set to multilingual E5, and the LLM is the best one
> you have in Ollama. First, the documents.

## 1. Load the documents, with metadata

Metadata is what lets you cite a source, filter by date, and exclude superseded rules. Pull it out as you load. For the
circulars, the header lines carry the issue date, the effective date, and what they supersede.

Look at the top of one circular to see what those header lines look like:

In [ ]:
# Look at the top of one circular: the header lines look like "**Date issued:** 12 September 2025".
circular_text = Path("circulars/FC_09_2025_penalty_update.md").read_text(encoding="utf-8")
print(circular_text[:500])

A small function reads one header field. The regular expression `\*\*Name:\*\*\s*(.+)` means: two asterisks, the field
name, a colon, two asterisks (the `\*` is a literal asterisk, since `*` on its own has a special meaning), any spaces,
then **capture** the rest of the line. `match.group(1)` is the captured part, the value.

In [ ]:
def field(text, name):
    """Read the value of a '**Name:** value' header line, or None if there isn't one."""
    match = re.search(r"\*\*" + name + r":\*\*\s*(.+)", text)
    if match is None:
        return None
    return match.group(1).strip()


print(field(circular_text, "Date issued"))
print(field(circular_text, "Supersedes"))

Now load every document into a list of dictionaries, one per document:

- **handbook sections:** the title is the first line without its `# `; the whole edition is dated 1 January 2026;
- **circulars:** the date that matters is when the rule takes **effect** (falling back to the issue date if there's no
  effective date), plus what the circular supersedes and its status. `pd.to_datetime(...).date()` turns "1 October
  2025" into a standard date, written as `2025-10-01`.

In [ ]:
docs = []

# Handbook sections (skipping 00_, the read-me-first note, which isn't policy).
for path in sorted(Path("handbook").glob("*.md")):
    if path.name.startswith("00_"):
        continue
    text = path.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()          # the first line, without the "# "
    docs.append({"source": path.name, "doc_type": "handbook", "title": title, "date": "2026-01-01", "text": text})

# Circulars, with their dates and what they supersede.
for path in sorted(Path("circulars").glob("*.md")):
    text = path.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()

    # The date that matters is when the rule takes effect; if that's missing, when it was issued.
    effective = field(text, "Effective")
    if effective:
        date = str(pd.to_datetime(effective).date())
    else:
        date = str(pd.to_datetime(field(text, "Date issued")).date())

    docs.append({"source": path.name, "doc_type": "circular", "title": title, "date": date,
                 "supersedes": field(text, "Supersedes"), "status": field(text, "Status"), "text": text})

pd.DataFrame(docs)[["source", "doc_type", "title", "date", "supersedes", "status"]]

The handbook edition is given a date of 1 January 2026 here, but notice that its penalty section *says* it's been updated
by circulars. The dates matter in Section 5.

> **Where we are.** Every handbook section and circular is loaded with its metadata: type, title, date, and for circulars
> what they supersede. The documents are still whole. The index needs smaller pieces.

## 2. Chunking

The index stores passages, not whole documents. Why not whole documents? Because a 10-page handbook section about
pensions matches almost any pension question a little, and handing all 10 pages to the LLM wastes its context and buries
the one sentence that matters. So documents are cut into **chunks**.

Size matters. A chunk that's too big dilutes the match and wastes the LLM's context. One that's too small loses the
sentence that gives it meaning. Two approaches:

- **Fixed size:** every 500 characters, overlapping by 100. Simple, blind to structure.
- **By structure:** one chunk per handbook sub-section (`## 4.3 How the pension is calculated`) and one per numbered circular
  paragraph. Each chunk keeps its heading, so it makes sense on its own.

The fixed-size version first. `range(0, len(text), size - overlap)` gives the start positions 0, 400, 800 ... so each
500-character chunk overlaps the previous one by 100 characters. The overlap means a sentence cut at one chunk's edge
still appears whole in the next.

In [ ]:
def chunk_fixed(doc, size=500, overlap=100):
    """Cut a document every `size` characters, each chunk overlapping the previous one by `overlap`."""
    text = doc["text"]
    chunks = []
    for start in range(0, len(text), size - overlap):          # 0, 400, 800, ...
        chunks.append({
            "source": doc["source"],
            "section": "chars " + str(start) + "-" + str(start + size),
            "text": text[start:start + size],
        })
    return chunks


fixed_chunks = []
for doc in docs:
    fixed_chunks.extend(chunk_fixed(doc))      # extend adds all of this document's chunks to the list
print(len(fixed_chunks), "fixed-size chunks")
print(fixed_chunks[0]["text"])

For structured chunks, a **regular expression** (Day 6) finds the places to cut. In the handbook, every sub-section starts
with `## ` at the beginning of a line. In a circular, every paragraph starts with a number and a full stop (`1. `).

Two functions, one per document type:

- `chunk_handbook` splits before every `## ` line. `(?=## )` is a lookahead (Day 6): it cuts *before* the `## ` without
  removing it. Each chunk starts with the section's title line (e.g. "# 4. Old Age Pension"), so a chunk about "4.3 How the
  pension is calculated" still says which benefit it's about;
- `chunk_circular` finds each numbered paragraph. The pattern `^(\d+)\.\s(.+?)(?=^\d+\.\s|\Z)` means: at the start of a
  line, a number, a full stop and a space; then as little text as possible (`.+?`) up to the next numbered line or the
  end of the text (`\Z`). `re.S` lets `.` match new lines too; `re.M` lets `^` match at the start of every line. Each chunk
  starts with the circular's title and effective date.

Both keep the document's type and date with every chunk: that metadata is used for filtering in Section 5.

In [ ]:
def chunk_handbook(doc):
    """One chunk per '## ' sub-section. Each chunk starts with the section title so it makes sense on its own."""
    # Split before every line that starts with "## ". (?=...) cuts there without removing the "## ".
    parts = re.split(r"\n(?=## )", doc["text"])
    section_title = parts[0].strip()             # e.g. "# 4. Old Age Pension"

    sub_sections = parts[1:]
    if len(sub_sections) == 0:                   # a section with no "## " sub-sections: keep it whole
        sub_sections = [doc["text"]]

    chunks = []
    for part in sub_sections:
        heading = part.splitlines()[0].lstrip("# ").strip()      # e.g. "4.3 How the pension is calculated"
        chunks.append({
            "source": doc["source"],
            "section": heading.split(" ")[0],                     # e.g. "4.3"
            "heading": heading,
            "text": section_title + "\n" + part.strip(),
            "doc_type": doc["doc_type"],
            "date": doc["date"],
        })
    return chunks


def chunk_circular(doc):
    """One chunk per numbered paragraph. Each chunk starts with the circular's title and effective date."""
    circular_id = re.search(r"FC/\d+/\d{4}", doc["title"]).group(0)      # e.g. "FC/09/2025"

    # A paragraph: a number at the start of a line, ". ", then everything up to the next numbered line or the end.
    paragraph_pattern = r"^(\d+)\.\s(.+?)(?=^\d+\.\s|\Z)"
    chunks = []
    for match in re.finditer(paragraph_pattern, doc["text"], flags=re.S | re.M):
        number = match.group(1)
        body = match.group(2).strip()
        chunks.append({
            "source": doc["source"],
            "section": circular_id + " para " + number,
            "heading": doc["title"],
            "text": doc["title"] + " (effective " + doc["date"] + ")\n" + number + ". " + body,
            "doc_type": doc["doc_type"],
            "date": doc["date"],
        })
    return chunks


# Chunk every document with the function for its type.
chunks = []
for doc in docs:
    if doc["doc_type"] == "handbook":
        chunks.extend(chunk_handbook(doc))
    else:
        chunks.extend(chunk_circular(doc))

print(f"fixed-size: {len(fixed_chunks)} chunks | structured: {len(chunks)} chunks")
print("\nOne structured chunk:\n", chunks[12]["text"])

> **Where we are.** You have two sets of chunks: fixed-size pieces with overlap, and one chunk per sub-section or numbered
> paragraph, each carrying its heading and date. Now turn them into something searchable by meaning.

### Theory: from word vectors to sentence embeddings

On Day 6, Word2Vec gave each **word** a vector, and averaging them made a rough vector for a message. A **sentence
embedding model** is a transformer trained to give a whole **passage** one vector, so that passages with the same meaning
land close together. It's trained on millions of pairs that should match (a question and its answer, a sentence and its
translation) and pairs that shouldn't, and learns to pull the matching pairs together and push the others apart.

Because it reads the whole passage with attention (Day 6), it captures meaning beyond single words: "How long is
maternity benefit paid?" and "Fao la uzazi linalipwa kwa muda gani?" can end up close together even though they
share no words. Searching then becomes geometry: embed the question, find the nearest passage vectors.

## 3. Embeddings and a vector index

An embedding model turns each chunk into a vector, so that passages with similar *meaning* end up close together, even
when they share no words. That's the big difference from Day 6's TF-IDF: "How long is maternity leave paid?" and "Fao la
uzazi linalipwa kwa muda gani?" share no words at all, but they mean the same thing, and a good multilingual embedding
model puts them close together.

### First, check the model actually knows Swahili

Many embedding models call themselves "multilingual". That means they were trained on a *particular list* of languages,
and Swahili isn't always well represented on it. Check before you build anything: a Swahili question should be closer to
the English question with the same meaning than to an unrelated one.

Each test is a triple: a Swahili question, the same question in English, and an unrelated English question. For each of
two candidate models, embed all three and check whether the Swahili question is more similar to its translation than to
the unrelated question. `normalize_embeddings=True` scales every vector to length 1, so similarity is purely about
direction. `del model` frees the memory before loading the next one.

In [ ]:
# (Swahili question, the same question in English, an unrelated English question)
PAIRS = [
    ("Fao la uzazi linalipwa kwa muda gani?", "How long is maternity benefit paid for?", "How much is the funeral grant?"),
    ("Msaada wa mazishi ni kiasi gani?", "How much is the funeral grant?", "What is the retirement age?"),
    ("Umri wa kustaafu ni miaka mingapi?", "What is the retirement age?", "How long is maternity benefit paid for?"),
]

# Two embedding models to compare. E5 expects the label "query: " in front of questions; the other doesn't.
CANDIDATE_MODELS = [
    ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ""),
    ("intfloat/multilingual-e5-small", "query: "),
]

for model_name, prefix in CANDIDATE_MODELS:
    model = SentenceTransformer(model_name, device=device)
    right = 0
    for swahili, same_english, other_english in PAIRS:
        vectors = model.encode([prefix + swahili, prefix + same_english, prefix + other_english], normalize_embeddings=True)
        similarity_same = float(util.cos_sim(vectors[0], vectors[1]))      # Swahili vs its translation
        similarity_other = float(util.cos_sim(vectors[0], vectors[2]))     # Swahili vs the unrelated question
        if similarity_same > similarity_other:
            right += 1
    print(f"{model_name.split('/')[-1]:42s} matched the right English question for {right} of {len(PAIRS)} Swahili questions")
    del model

On the test run, the popular `paraphrase-multilingual-MiniLM-L12-v2` model got at least one of these wrong, and even its
correct matches had very low similarity scores. Swahili isn't among the languages it was trained on, so a Swahili question
and its English translation look almost unrelated to it. The E5 model, built on a base trained on about 100 languages
including Swahili, matched all three. Five minutes of checking like this saves you from a system that quietly fails a large
share of members. Three pairs is only a smoke test; before choosing a model for real, use a few dozen.

E5 models expect every text to start with a label: `query: ` for questions and `passage: ` for the documents being searched.
Leaving the labels out quietly lowers quality.

FAISS (Facebook AI Similarity Search, a library for finding nearest vectors fast) stores the vectors and finds the nearest
ones quickly. With a few hundred chunks any method is fast. With the Fund's full document library (hundreds of thousands of
passages) FAISS still answers in milliseconds.

### What an embedding looks like

The embedding model turns any piece of text into a list of 384 numbers. On their own the numbers mean nothing to a
person. What matters is that texts with similar meaning get similar lists:

```
"How long is maternity benefit paid?"    ->  [ 0.031, -0.112,  0.087, ... ]   (384 numbers)
"Fao la uzazi linalipwa kwa muda gani?"   ->  [ 0.029, -0.105,  0.091, ... ]   close to the line above
"What is the retirement age?"             ->  [-0.064,  0.020, -0.013, ... ]   further away
```

Finding the passages that answer a question is then a geometry problem: find the stored lists closest to the
question's list. That's what FAISS does, very quickly.

Load the chosen model and embed one question to see it for yourself: the shape is (1 text, 384 numbers).

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL, device=device)

# One example: a question becomes 384 numbers.
example_vector = embedder.encode(["query: How long is maternity benefit paid?"], normalize_embeddings=True)
print(example_vector.shape)
print(example_vector[0, :6].round(3))       # the first 6 of the 384 numbers

`build_index` embeds every chunk (with the `passage: ` label) and stores the vectors in a FAISS index. `IndexFlatIP`
is the simplest kind of index: it compares the question with every stored vector by **inner product** (multiply
matching numbers and add up). For vectors of length 1, that's exactly the cosine similarity. `batch_size=64` embeds 64
chunks at a time. Build one index for each chunking method, so you can compare them.

In [ ]:
def build_index(chunk_list):
    """Embed every chunk and store the vectors in a FAISS index."""
    texts = []
    for chunk in chunk_list:
        texts.append("passage: " + chunk["text"])

    vectors = embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

    # IndexFlatIP compares vectors by "inner product", which equals cosine similarity for normalised vectors.
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors.astype(np.float32))
    return index


start = time.perf_counter()
index = build_index(chunks)                   # structured chunks
fixed_index = build_index(fixed_chunks)       # fixed-size chunks
print(f"embedded {len(chunks) + len(fixed_chunks)} chunks in {time.perf_counter() - start:.1f}s; vector size {index.d}")

**Dense search**: embed the question (with the `query: ` label), ask FAISS for the `k` nearest chunk vectors, and return
each chunk with its similarity. FAISS returns two lists: the similarity scores, and the **positions** of the matching
vectors, which are also the positions of the chunks in `chunk_list` (the vectors were added in the same order).

Try it with the Swahili maternity question:

In [ ]:
def dense_search(question, k=5, idx=index, chunk_list=chunks):
    """The k chunks whose meaning is closest to the question. Returns a list of (chunk, similarity) pairs."""
    question_vector = embedder.encode(["query: " + question], normalize_embeddings=True).astype(np.float32)

    # FAISS returns the similarities and the positions of the k nearest vectors.
    scores, positions = idx.search(question_vector, k)

    results = []
    for position, score in zip(positions[0], scores[0]):
        results.append((chunk_list[position], float(score)))
    return results


for chunk, score in dense_search("Fao la uzazi linalipwa kwa muda gani?", k=3):
    print(f"{score:.3f}  {chunk['source']:32s} {chunk['heading']}")

A Swahili question about maternity benefit ("how long is it paid for?") should find the English maternity section.
Notice how close together the scores are: E5 models give most related texts a similarity between about 0.7 and 0.9. What
matters is the *ranking*, not the raw number, which is worth remembering when you set a threshold in Section 6.

> **Where we are.** Both chunk sets are embedded and stored in FAISS indexes, and `dense_search` finds the passages
> closest in meaning to any question. A Swahili question finds the English maternity section. Before adding an LLM,
> measure how often retrieval finds the right document.

### Theory: test each stage separately

A RAG system is a chain: retrieval, then generation. When the final answer is wrong, you need to know **which link
broke**, because the fixes are completely different. So measure the links one at a time:

1. **Retrieval:** for each question, is the right document in the top *k*? (hit@k, MRR). No LLM needed, so it's fast and
   cheap to rerun after every change to chunking or search.
2. **Generation:** given the right passages, does the model answer correctly, cite them, and refuse when it should?
3. **End to end:** the whole assistant on every test question (Section 8).

The same principle runs through the course: check each stage of a pipeline on its own (Day 1), and know which part a
failure came from before you try to fix it.

## 4. Measure retrieval before adding an LLM

If the right passage isn't retrieved, no LLM can give the right answer. So test retrieval on its own first. For each
test question you know which document should answer it:

- **hit@k:** was a chunk from the right document among the top *k* results?
- **MRR** (mean reciprocal rank): 1 if it came first, 1/2 if second, and so on. It rewards putting the right passage at
  the top.

The test file has one row per question, with the document that should answer it (`expected_source`). Some questions have
no expected document: the documents don't answer them, and the right response is a refusal. Those are left out of the
retrieval test (there's nothing to find), but come back in Section 6.

In [ ]:
evalq = pd.read_csv("rag_eval_questions.csv")
evalq["expected_source"] = evalq["expected_source"].fillna("")       # empty = the documents don't answer it

answerable = evalq[evalq["expected_source"] != ""]
print(len(evalq), "test questions,", len(answerable), "answerable from the documents")

`evaluate_retrieval` takes any search function (anything you can call as `search(question, k)`) and measures it. For each
answerable question it lists the documents of the top `k` results, then checks whether the expected document is among
them, and at which position. `found_sources.index(expected)` gives that position (0 for first), so the reciprocal rank is
`1 / (position + 1)`.

In [ ]:
def evaluate_retrieval(search_function, k=5):
    """hit@k and MRR of a search function over the answerable test questions."""
    hits = []
    reciprocal_ranks = []
    for question, expected in zip(answerable["question"], answerable["expected_source"]):
        # The documents of the top k results, in order.
        found_sources = []
        for chunk, score in search_function(question, k):
            found_sources.append(chunk["source"])

        if expected in found_sources:
            hits.append(1)
            reciprocal_ranks.append(1 / (found_sources.index(expected) + 1))
        else:
            hits.append(0)
            reciprocal_ranks.append(0)
    return {"hit@k": np.mean(hits), "MRR": np.mean(reciprocal_ranks)}

Compare the two chunking methods. Two small functions give each index the same `(question, k)` shape, so
`evaluate_retrieval` can test both.

In [ ]:
# Two search functions with the same shape, (question, k), one per chunking method.
def search_fixed(question, k):
    return dense_search(question, k, idx=fixed_index, chunk_list=fixed_chunks)


def search_structured(question, k):
    return dense_search(question, k, idx=index, chunk_list=chunks)


pd.DataFrame({
    "fixed-size chunks": evaluate_retrieval(search_fixed),
    "structured chunks": evaluate_retrieval(search_structured),
}).T.round(3)

### Adding keyword search: hybrid retrieval

Embeddings are good at meaning and weak at exact tokens: a circular number like *FC/09/2025*, a figure like *20,000*, or a
rare term. Classic keyword search (TF-IDF, short for term frequency-inverse document frequency, here) is the opposite.
**Hybrid retrieval** runs both and merges the rankings.

First, keyword search on its own. Build a TF-IDF table of all the chunks (Day 6). To search, turn the question into the
same kind of vector and multiply: `chunk_matrix @ question_vector.T` gives one similarity number per chunk (`@` is
matrix multiplication). `np.argsort(-similarities)` lists the chunk positions from most to least similar (the minus sign
makes it sort high to low).

In [ ]:
# TF-IDF over the chunk texts (Day 6).
chunk_texts = []
for chunk in chunks:
    chunk_texts.append(chunk["text"])

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
chunk_matrix = tfidf.fit_transform(chunk_texts)


def keyword_search(question, k=5):
    """The k chunks that share the most distinctive words with the question (TF-IDF similarity)."""
    question_vector = tfidf.transform([question])
    similarities = (chunk_matrix @ question_vector.T).toarray().ravel()     # one number per chunk
    best = np.argsort(-similarities)[:k]                                     # positions of the k highest
    results = []
    for position in best:
        results.append((chunks[position], float(similarities[position])))
    return results


for chunk, score in keyword_search("FC/09/2025 penalty", k=3):
    print(f"{score:.3f}  {chunk['source']:32s} {chunk['section']}")

Now merge the two rankings. The scores from the two methods are on different scales, so don't add the scores; add
points based on **rank** instead. **Reciprocal rank fusion** is a simple, dependable way to do that: each chunk scores
1/(60 + rank) in each list, and the points add up. A chunk near the top of both lists gets the most points. (The 60 is a
standard constant that stops the very first place from dominating.)

Each chunk is identified by `(source, section)`. The function also remembers each chunk's dense similarity, which Section 6
uses for the relevance threshold. `points.get(key, 0.0)` reads a chunk's points so far, or 0 if it has none yet.

In [ ]:
def hybrid_search(question, k=5, pool=20):
    """Merge dense and keyword rankings with reciprocal rank fusion. Returns (chunk, dense similarity) pairs."""
    dense_results = dense_search(question, k=pool)
    keyword_results = keyword_search(question, k=pool)

    # Every chunk gets 1 / (60 + rank) points from each list it appears in.
    points = {}        # (source, section) -> points
    chunk_by_key = {}  # (source, section) -> the chunk itself
    for results in [dense_results, keyword_results]:
        for rank, (chunk, score) in enumerate(results):
            key = (chunk["source"], chunk["section"])
            chunk_by_key[key] = chunk
            points[key] = points.get(key, 0.0) + 1 / (60 + rank)

    # Each chunk's dense similarity (0 if dense search didn't find it). Used later for the relevance threshold.
    dense_similarity = {}
    for chunk, score in dense_results:
        dense_similarity[(chunk["source"], chunk["section"])] = score

    # The k chunks with the most points.
    best_keys = sorted(points, key=points.get, reverse=True)[:k]
    merged = []
    for key in best_keys:
        merged.append((chunk_by_key[key], dense_similarity.get(key, 0.0)))
    return merged


pd.DataFrame({
    "dense (embeddings)": evaluate_retrieval(dense_search),
    "keyword (TF-IDF)": evaluate_retrieval(keyword_search),
    "hybrid": evaluate_retrieval(hybrid_search),
}).T.round(3)

Look at which questions each method misses, not just the averages. A miss on the penalty question is far more serious than
a miss on branch opening hours.

**Your turn (1).** Print, for the hybrid search, every answerable question where the expected document is *not* the first
result. Why did retrieval rank something else first? Is it wrong, or did it find another document that also answers the
question?

> **Where we are.** You've measured both chunking methods with hit@k and MRR, and hybrid search (meaning plus keywords,
> merged by rank) adds the exact-token matches embeddings miss. Retrieval finds the right document for most questions.
> But finding *a* relevant passage isn't the same as finding the *current* one.

In [ ]:
# Your code here

## 5. Superseded rules

Ask about the late-payment penalty and look at what comes back. Each line shows the similarity, the section, its date
and the start of its text (`replace("\n", " ")` keeps it on one line).

In [ ]:
for chunk, score in hybrid_search("What is the penalty for paying contributions late?", k=5):
    first_line = chunk["text"][:110].replace("\n", " ")
    print(f"{score:.3f}  {chunk['section']:22s} {chunk['date']}  {first_line}")

The handbook's 5%, circular FC/07/2024 (5%, plus an amnesty), and circular FC/09/2025 (2.5%, capped at 50%) can all be
retrieved together, and they disagree. If you hand all of them to the LLM and hope it picks the newest, a small model will
sometimes pick the wrong one (you saw this on Day 7).

The reliable fix is to **resolve conflicts before generation**, using metadata. Record which passages have been
superseded, and filter them out of retrieval. `SUPERSEDED` maps each out-of-date passage, identified by
`(document, section)`, to the reason it no longer applies. The loop adds a `superseded_by` field to every chunk: the
reason if it's out of date, `None` if it's current.

In [ ]:
# (document, section) -> why it no longer applies
SUPERSEDED = {
    ("FC_07_2024_late_payment_penalty.md", "FC/07/2024 para 1"): "Superseded by FC/09/2025 para 1 (rate reduced to 2.5%)",
    ("FC_07_2024_late_payment_penalty.md", "FC/07/2024 para 2"): "Amnesty ended 31 Dec 2024; confirmed by FC/09/2025 para 4",
    ("03_contributions.md", "3.3"): "Rate superseded by FC/09/2025 from 1 October 2025",
    ("04_old_age_pension.md", "4.6"): "Verification methods updated by FC/03/2025",
}

# Mark each chunk: the reason if it's superseded, None if it's current.
n_superseded = 0
for chunk in chunks:
    chunk["superseded_by"] = SUPERSEDED.get((chunk["source"], chunk["section"]))
    if chunk["superseded_by"] is not None:
        n_superseded += 1
print(n_superseded, "chunks marked as superseded")

`current_search` is hybrid search with the superseded passages thrown away. It asks for 6 extra results first, so that
after removing the out-of-date ones there are still `k` left. Run the penalty question again: only the current rule
should come back.

In [ ]:
def current_search(question, k=5):
    """Hybrid search with superseded passages removed, so only current rules reach the LLM."""
    # Ask for a few extra results, because some will be thrown away.
    results = []
    for chunk, score in hybrid_search(question, k=k + 6):
        if chunk["superseded_by"] is None:
            results.append((chunk, score))
    return results[:k]


for chunk, score in current_search("What is the penalty for paying contributions late?", k=3):
    first_line = chunk["text"][:110].replace("\n", " ")
    print(f"{score:.3f}  {chunk['section']:22s} {first_line}")

Who maintains that `SUPERSEDED` table? In a real deployment it's not the data science team. It's whoever issues circulars.
The practical design is a **document register**: when a circular is published, the person publishing it records which
earlier rules it replaces, and the index is rebuilt. RAG makes this possible (update a document, rebuild, done), but it
doesn't make it automatic. It's a governance process with a technical step at the end.

Handbook section 4.6 is marked superseded too: circular FC/03/2025 added app verification and home visits. Removing it
entirely would lose the parts that still hold (12 months; suspended, not cancelled), but the circular repeats those
points, so nothing is lost here. On real documents, check before you exclude.

> **Where we are.** `current_search` returns only passages that are still in force; superseded ones are filtered out using
> the document register in `SUPERSEDED`. Retrieval is now both relevant and current. Time to add the LLM.

## 6. Generation with citations, and knowing when to say "I don't know"

Two safeguards around the LLM:

1. **A relevance threshold.** If even the best passage is only weakly related to the question, don't ask the LLM at all.
   Return the refusal directly. The LLM can't invent an answer it was never asked for.
2. **Citations.** The passages are numbered, and the model must cite the numbers it used. That lets a staff member check
   the answer in seconds.

First look at how the best retrieval score differs between questions that *can* be answered and questions that can't. The
threshold goes between them. For every test question, take the similarity of its single best chunk, and sort with the
lowest at the top:

In [ ]:
rows = []
for question, expected in zip(evalq["question"], evalq["expected_source"]):
    best_chunk, best_score = dense_search(question, k=1)[0]          # the top result and its similarity
    rows.append({"question": question, "answerable": expected != "", "best dense score": best_score})

best = pd.DataFrame(rows)
best.sort_values("best dense score").head(8)

Set the threshold just above the best score of any **unanswerable** question (`~best["answerable"]` = the questions
that aren't answerable), plus a small margin of 0.02. Then count how many *answerable* questions fall below it: those
would be wrongly refused.

In [ ]:
# Just above the best-scoring unanswerable question.
MIN_SCORE = float(best.loc[~best["answerable"], "best dense score"].max() + 0.02)

blocked = (best.loc[best["answerable"], "best dense score"] < MIN_SCORE).sum()
print(f"relevance threshold: {MIN_SCORE:.2f}; answerable questions below it: {blocked}")

Look at how thin the margin is: with E5 the best unanswerable question scores only a few hundredths below the weakest
answerable one. With only two unanswerable test questions, this threshold is a first guess, not a calibrated value. Before
production you'd set it using a few hundred real questions, including plenty that the documents *don't* answer, and you'd
check how many good questions it wrongly blocks.

Now the instructions for the LLM. The system prompt says: answer only from the numbered sources, cite them like `[1]`,
use the exact refusal sentence if the sources don't answer the question, quote figures exactly, keep it short, and reply
in the question's language. `generate` sends one prompt with those rules, at temperature 0 (Day 7).

In [ ]:
REFUSAL = "I can't find this in the Fund's documents. Please contact the call centre or visit a branch."

SYSTEM = """You are the Fund's handbook assistant for staff. Answer ONLY from the numbered sources.
- Cite the sources you used in square brackets, e.g. [1] or [2][3].
- If the sources do not contain the answer, reply exactly: """ + REFUSAL + """
- Quote figures exactly as they appear. Keep the answer under 80 words.
- Reply in the language of the question."""


def generate(prompt, max_tokens=250):
    """Send the sources and question to the local LLM with the strict answering rules."""
    body = {
        "model": LLM,
        "stream": False,
        "options": {"temperature": 0, "num_predict": max_tokens, "seed": 42},
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}],
    }
    reply = requests.post(OLLAMA_URL + "/api/chat", json=body, timeout=300).json()
    return reply["message"]["content"].strip()

The full pipeline, in five steps:

1. **retrieve** the current passages (superseded ones removed);
2. **check the threshold**: if even the best passage is too weak, return the refusal without asking the LLM;
3. **number the passages** `[1]`, `[2]` ..., each with its document, section and date, so the model can cite them;
4. **ask the LLM**;
5. **read the citations**: `re.findall(r"\[(\d+)\]", answer)` finds every `[n]` in the answer. Keep only numbers that
   match a real passage (the model sometimes invents `[7]` when there were 4), turn them into document names, and return
   them with the answer.

`enumerate(results, start=1)` numbers the passages from 1 rather than 0. The `route` field records which path the
question took, which is useful when checking why an answer came out as it did.

In [ ]:
def rag_answer(question, k=4):
    """The full pipeline: retrieve, refuse if nothing is relevant enough, otherwise answer from numbered sources."""
    # 1. Retrieve current passages.
    results = current_search(question, k=k)

    # 2. Relevance threshold: if even the best passage is too weak, refuse without asking the LLM.
    best_score = 0.0
    for chunk, score in results:
        best_score = max(best_score, score)
    if len(results) == 0 or best_score < MIN_SCORE:
        return {"answer": REFUSAL, "sources": [], "route": "below relevance threshold"}

    # 3. Number the passages: [1] (source, section, date) text ...
    context = ""
    for number, (chunk, score) in enumerate(results, start=1):
        context = (context + "[" + str(number) + "] (" + chunk["source"] + ", " + chunk["section"] + ", dated "
                   + chunk["date"] + ")\n" + chunk["text"] + "\n\n")

    # 4. Ask the LLM.
    answer = generate("Sources:\n" + context + "Question: " + question)

    # 5. Which numbers did it cite? Find every [n] in the answer, keep the valid ones, without repeats.
    cited_numbers = set()
    for found in re.findall(r"\[(\d+)\]", answer):
        n = int(found)
        if 1 <= n <= len(results):
            cited_numbers.add(n)

    sources = []
    for n in sorted(cited_numbers):
        chunk = results[n - 1][0]                    # [n] is position n - 1 in the list
        sources.append(chunk["source"] + " " + chunk["section"])

    return {"answer": answer, "sources": sources, "route": "answered"}


rag_answer("What is the penalty for paying contributions late?")

> **Where we are.** `rag_answer` retrieves current passages, refuses without asking the LLM if nothing is relevant enough,
> numbers the passages, asks for an answer with citations, and keeps only real citation numbers. One kind of question
> must never reach it at all.

### Theory: layers of defence

No single safeguard is enough, so the assistant has several, each catching what the others miss:

```
question ─► guardrail (personal records?) ─► retrieval (current passages only) ─► threshold (relevant enough?)
         ─► LLM (answer only from sources, cite them) ─► citation check (real source numbers only) ─► answer
```

The checks **outside** the model (guardrail, superseded filter, threshold, citation check) are ordinary code. They behave
the same way every time and can't be argued with by a cleverly worded question. The instructions **inside** the prompt
make good behaviour likely but not certain. A safe design puts the most important rules outside the model.

## 7. A guardrail for personal-record questions

On Day 8 you saw a model invent a member's balance, even with the handbook in front of it. Personal records are never in
the handbook, so these questions should never reach the LLM. A simple rule catches most of them, in both languages, and
returns a safe, useful redirect.

The pattern looks for phrases like "my balance", "my statement", "my claim", "my member number" and their Swahili
equivalents (*salio langu*, *madai yangu*, *michango yangu*). `re.compile` prepares the pattern once so it can be used
many times; `re.IGNORECASE` ignores capitalisation. `assistant` is the **front door**: it checks the guardrail first, and
only sends the question on to `rag_answer` if the guardrail doesn't fire.

In [ ]:
# "my balance", "my statement", "my claim", "my member number", ... and the Swahili equivalents.
PERSONAL = re.compile(
    r"\bmy (balance|statement|claim|member (number|no)|contributions? (balance|history|record)|pension (status|payment))\b"
    r"|\bsalio (langu|la michango)|\bnamba yangu ya uanachama|\bmadai yangu|\bmichango yangu",
    re.IGNORECASE,
)
PERSONAL_REPLY = ("I can't see personal records. Members can check balances, statements and claim status on the member "
                  "portal, the mobile app, the USSD service, or at any branch with their NIDA card.")


def assistant(question):
    """The front door: personal-record questions get the safe redirect; everything else goes to rag_answer."""
    if PERSONAL.search(question):
        return {"answer": PERSONAL_REPLY, "sources": [], "route": "personal-data guardrail"}
    return rag_answer(question)


for question in ["What is my contribution balance?", "Naomba kujua salio la michango yangu", "How much is the funeral grant?"]:
    print(question, "->", assistant(question))
    print()

The guardrail is a regular expression. It's crude, it's transparent, and it's reliable for what it covers. It will miss
some phrasings ("how much have I saved with you?"), so keep a list of the misses you find and extend it. And because it runs
*before* the LLM, it can't be talked out of its job by a clever prompt.

> **Where we are.** The assistant is complete: guardrail first, then retrieval, threshold, LLM and citation check. Now test
> it the way members will use it, on every test question.

## 8. End-to-end evaluation

Now test the whole assistant (guardrail, retrieval, threshold, LLM) on every test question. An answer **passes** if it
contains the question's key fact or, for a question the documents don't answer, if it refuses ("can't find").

In [ ]:
def passes(answer, must_include, expected_source):
    """True if the reply contains the key fact, or (for questions the documents don't answer) refuses."""
    text = answer.lower().replace(",", "")
    if expected_source == "":
        return "can't find" in text or "cannot find" in text
    return must_include.lower().replace(",", "") in text

For every test question, record which route it took, the answer, the sources cited, whether it passed, and whether it
cited the document it should have. (For an unanswerable question, citing nothing counts as correct.)

In [ ]:
rows = []
start = time.perf_counter()

for question, expected, must in zip(evalq["question"], evalq["expected_source"], evalq["must_include"]):
    out = assistant(question)

    # Did it cite the document it should have? (For unanswerable questions: did it cite nothing?)
    if expected == "":
        cited_expected = len(out["sources"]) == 0
    else:
        cited_expected = False
        for source in out["sources"]:
            if expected in source:
                cited_expected = True

    rows.append({
        "question": question,
        "route": out["route"],
        "answer": out["answer"],
        "sources": "; ".join(out["sources"]),
        "expected_source": expected,
        "correct": passes(out["answer"], must, expected),
        "cited_expected": cited_expected,
    })

e2e = pd.DataFrame(rows)
print(f"{len(e2e)} questions in {time.perf_counter() - start:.0f}s")
print(f"answer contains the key fact (or correctly refuses): {e2e['correct'].mean():.0%}")
print(f"cites the expected document:                      {e2e['cited_expected'].mean():.0%}")
e2e[["question", "route", "correct", "cited_expected", "answer"]]

Compare this with Day 8, where the best system answered about a quarter of held-out questions correctly. Retrieval
plus a strict prompt plus a guardrail makes a far bigger difference than fine-tuning did, with the same kind of small model.

Look through the failures. Each will fall into one of three buckets, and each bucket has a different fix:

| Failure | Where to look | Typical fix |
|---|---|---|
| Right passage never retrieved | Section 4 | Better chunking, hybrid search, more `k` |
| Retrieved, but the model answered wrongly or didn't cite | The prompt, the model | Stricter prompt, bigger model (Day 7 scorecard) |
| Refused when it shouldn't, or answered when it should have refused | The threshold, the guardrail | Calibrate on more questions |

Also read the answers that *passed*. Keyword scoring is a quick filter, not a verdict. On the test run, the one "failure"
was a correct answer ("benefits are only paid to the member or a verified dependant") that simply didn't use the words
"own name". And one "pass" was misleading: *"Yes, workers in Zanzibar are covered by the Zanzibar scheme"* contains the right
fact but starts with a "Yes" that a busy reader would take as "yes, the Fund covers them". Before go-live, people who answer
members' questions every day should read and grade a large sample of answers.

**Your turn (2).** Pick one failed question and fix it, using the table above to decide where. Rerun the evaluation. Did the
fix break anything else?

In [ ]:
# Your code here

> **Where we are.** The whole assistant has been tested end to end, and you know which bucket each failure falls into. It
> works in this notebook. To be useful, other systems need to be able to call it.

## 9. Save the index for Lab 2

Lab 2 turns this pipeline into a web service, so it needs everything built here, saved to a `rag_index` folder:

1. the FAISS index (the vectors), with `faiss.write_index`;
2. the chunks with all their metadata, as JSON, **in the same order as the vectors** (position 7 in the index must be
   chunk 7);
3. every setting that affects the answers: the models, the threshold, `k`, the refusal and redirect texts, the guardrail
   pattern, the system prompt and the build time.

In [ ]:
out = Path("rag_index")
out.mkdir(exist_ok=True)

# 1. The vectors.
faiss.write_index(index, str(out / "chunks.faiss"))

# 2. The chunks, with their metadata, in the same order as the vectors.
(out / "chunks.json").write_text(json.dumps(chunks, ensure_ascii=False, indent=1), encoding="utf-8")

# 3. Every setting that affects the answers.
config = {
    "embed_model": EMBED_MODEL,
    "llm": LLM,
    "min_score": MIN_SCORE,
    "k": 4,
    "refusal": REFUSAL,
    "personal_reply": PERSONAL_REPLY,
    "personal_pattern": PERSONAL.pattern,
    "system_prompt": SYSTEM,
    "built": time.strftime("%Y-%m-%d %H:%M"),
}
(out / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

for path in sorted(out.iterdir()):
    print("saved:", path)

The index, the chunks with their metadata, and every setting that affects answers are saved together. When someone asks why
the assistant said something on a given day, `config.json` and the chunks tell you exactly what it knew.

---

## Looking back, looking ahead

**What you did in this lab.** You built a RAG pipeline stage by stage: documents with metadata, two chunking methods,
embeddings checked for Swahili, a FAISS index, retrieval measured on its own, hybrid search with reciprocal rank fusion,
superseded rules filtered out before generation, answers with citations behind a relevance threshold, a guardrail for
personal records, an end-to-end evaluation, and a saved index with every setting.

**The ideas to carry forward.**

- RAG changes the model's job from remembering to reading.
- Most RAG failures are retrieval failures; measure retrieval first.
- Resolve outdated rules with metadata before the LLM sees them; keep a document register.
- Put the most important safeguards outside the model.
- Keyword scores are a filter; people grade the answers before go-live.

**What happens next.** Lab 2 puts this pipeline behind an internal API with authentication, rate limits and logging, and
calls it from a small in-house app.

## Suggested answer to (1)

For each answerable question, look at the top 3 hybrid results; where the first one isn't the expected document, print
what came back instead:

In [ ]:
for question, expected in zip(answerable["question"], answerable["expected_source"]):
    top3 = hybrid_search(question, k=3)
    first_source = top3[0][0]["source"]          # the document of the first result
    if first_source != expected:
        found = []
        for chunk, score in top3:
            found.append(chunk["source"] + " " + chunk["section"])
        print("Q:", question)
        print("   expected", expected, "; got:", ", ".join(found))